# 06 - Composable Extraction Pipeline

This notebook demonstrates how to customize the extraction pipeline — the process that turns your documents into entities, relationships, and topics for the knowledge graph.

By default, lexical-graph runs a fixed two-step extraction (propositions → topics). The composable pipeline lets you define your own sequence of stages, add schema constraints, combine LLM extraction with local NER models, and write custom processing steps.

## Setup

If you haven't already, install the toolkit and dependencies using the [Setup](./00-Setup.ipynb) notebook.

In [ ]:
# Install optional dependency for NER stage (Example 4)
# Skip this cell if you don't plan to use NER
%pip install -q gliner

In [ ]:
%reload_ext dotenv
%dotenv

import os

from graphrag_toolkit.lexical_graph import LexicalGraphIndex, set_logging_config
from graphrag_toolkit.lexical_graph.lexical_graph_index import ExtractionConfig
from graphrag_toolkit.lexical_graph.indexing.extract import (
    PipelineBuilder,
    ExtractionSchema,
    EntityTypeConfig,
    LLMPropositionStage,
    LocalPropositionStage,
    LLMTopicExtractionStage,
    SchemaFilterStage,
    NERExtractionStage,
    EntityMergeStage,
)

set_logging_config('INFO')

## Example 1: Pipeline Variations

The simplest use — pick which stages run and in what order. Three common patterns:

In [ ]:
# Pattern A: Default — same as built-in behavior, but explicit
default_pipeline = ExtractionConfig(
    stages=[
        LLMPropositionStage(),
        LLMTopicExtractionStage(),
    ]
)

# Pattern B: Skip propositions — extract topics directly from text (faster, cheaper)
direct_pipeline = ExtractionConfig(
    stages=[
        LLMTopicExtractionStage(use_propositions=False),
    ]
)

# Pattern C: Local propositions — use a local model instead of LLM (free, no API calls)
local_pipeline = ExtractionConfig(
    stages=[
        LocalPropositionStage(),
        LLMTopicExtractionStage(),
    ]
)

for name, config in [('Default', default_pipeline), ('Direct', direct_pipeline), ('Local', local_pipeline)]:
    stages = [type(s).__name__ for s in config.stages]
    print(f'{name:8s} → {" → ".join(stages)}')

## Example 2: Schema-Constrained Extraction

Define which entity types and relationship types you care about. When `schema` is passed to
`ExtractionConfig`, it is automatically injected into the LLM prompt — guiding extraction, not
just filtering. The `SchemaFilterStage` then removes anything the LLM still extracted outside the schema.

In [ ]:
schema = ExtractionSchema(
    entity_types={
        'Person': EntityTypeConfig(
            description='A human individual',
            aliases=['Individual', 'Human'],  # These also match
        ),
        'Company': EntityTypeConfig(
            description='A business organization',
            aliases=['Organization', 'Corp'],
        ),
    },
    relationship_types=['WORKS_FOR', 'FOUNDED', 'ACQUIRED'],
    strict=True,  # Enforce filtering
)

# See what the schema looks like as a prompt constraint
print('Prompt constraint that gets injected into the LLM:\n')
print(schema.format_as_prompt_constraint())

# Use it in a pipeline
extraction_config = ExtractionConfig(
    stages=[
        LLMPropositionStage(),
        LLMTopicExtractionStage(),
        SchemaFilterStage(schema),
    ],
    schema=schema,
)

print(f'\nAllowed entity types: {schema.entity_type_names()}')
print(f'Allowed relationships: {schema.relationship_types}')

## Example 3: Pipeline Validation

The `PipelineBuilder` checks that each stage's required inputs are satisfied by prior stages' outputs. This catches wiring mistakes before you spend time and money on LLM calls.

In [ ]:
# Valid pipeline — propositions come before topic extraction
builder = PipelineBuilder()
builder.add(LLMPropositionStage())
builder.add(LLMTopicExtractionStage())
transforms = builder.build()
print(f'✅ Valid pipeline: {len(transforms)} transforms')
print(f'   Available keys after build: {builder.available_keys}')

# Invalid pipeline — topic extraction needs propositions but none provided
print()
try:
    PipelineBuilder().add(LLMTopicExtractionStage()).build()
except ValueError as e:
    print(f'❌ Caught wiring error: {e}')

## Example 4: NER + LLM Hybrid Extraction

Augment LLM extraction with a local NER model ([GLiNER](https://github.com/urchade/GLiNER)). NER runs on CPU, costs nothing, and catches entities the LLM might miss. The `EntityMergeStage` combines both sources and deduplicates.

In [ ]:
schema = ExtractionSchema(
    entity_types={
        'Person': EntityTypeConfig(aliases=['Individual']),
        'Company': EntityTypeConfig(aliases=['Organization', 'Corp']),
        'Technology': EntityTypeConfig(),
    },
    relationship_types=['WORKS_FOR', 'USES', 'DEVELOPS', 'ACQUIRED'],
    strict=True,
)

extraction_config = ExtractionConfig(
    stages=[
        LLMPropositionStage(),                                              # 1. Break text into propositions
        NERExtractionStage(                                                 # 2. Find entities locally (free)
            entity_labels=['Person', 'Organization', 'Technology'],
            threshold=0.5,
        ),
        LLMTopicExtractionStage(),                                          # 3. Extract topics via LLM
        EntityMergeStage(fuzzy_threshold=0.85),                              # 4. Merge NER + LLM entities (fuzzy dedup)
        SchemaFilterStage(schema),                                          # 5. Keep only what we want
    ],
    schema=schema,
)

print('Pipeline:')
for i, stage in enumerate(extraction_config.stages):
    print(f'  {i+1}. {type(stage).__name__:30s} type={stage.stage_type:10s} in={stage.input_keys()}  out={stage.output_keys()}')

## Example 5: Custom Stage

Implement the `ExtractionStage` ABC to add your own processing logic — sentiment analysis, language detection, external NER injection, or anything else.

In [ ]:
from typing import List, Sequence
from llama_index.core.schema import BaseNode, TransformComponent
from graphrag_toolkit.lexical_graph.indexing.extract import ExtractionStage

STATS_KEY = 'my::text_stats'

class TextStatsTransform(TransformComponent):
    """Adds word/char counts to node metadata."""
    def __call__(self, nodes: Sequence[BaseNode], **kwargs) -> Sequence[BaseNode]:
        for node in nodes:
            text = node.get_content()
            node.metadata[STATS_KEY] = {'words': len(text.split()), 'chars': len(text)}
        return nodes

class TextStatsStage(ExtractionStage):
    def input_keys(self) -> List[str]:
        return []
    def output_keys(self) -> List[str]:
        return [STATS_KEY]
    def as_transform(self) -> TransformComponent:
        return TextStatsTransform()
    @property
    def stage_type(self) -> str:
        return 'custom'

# Plug it into a pipeline
builder = PipelineBuilder()
builder.add(TextStatsStage())
builder.add(LLMPropositionStage())
builder.add(LLMTopicExtractionStage())
transforms = builder.build()

print(f'Pipeline with custom stage: {len(transforms)} transforms')
print(f'Available keys: {builder.available_keys}')

## Example 6: End-to-End — NER + Schema Filtering

Put it all together with a real file. We load `artifacts/sample.md` (a markdown formatting guide
mentioning tools like Pandoc and Python). NER and schema are aligned on the same entity types,
with the schema doing light filtering — keeping the three main types while dropping anything
the LLM might hallucinate outside those categories.

In [ ]:
from llama_index.core import SimpleDirectoryReader

# Schema aligned with NER labels — both target the same entity types
# Light filtering: keeps Software, Format, and Concept; drops anything else
schema = ExtractionSchema(
    entity_types={
        'Software': EntityTypeConfig(
            description='A software tool, language, or library',
            aliases=['Tool', 'Language', 'Library', 'Program'],
        ),
        'Format': EntityTypeConfig(
            description='A file format, markup language, or data notation',
            aliases=['Markup', 'Syntax', 'Notation'],
        ),
        'Concept': EntityTypeConfig(
            description='A technical concept or standard',
            aliases=['Standard', 'Feature', 'Specification'],
        ),
    },
    relationship_types=['SUPPORTS', 'CONVERTS_TO', 'USES', 'PART_OF', 'RELATED_TO'],
    strict=True,
)

extraction_config = ExtractionConfig(
    stages=[
        LLMPropositionStage(),
        NERExtractionStage(
            # Same types as schema — NER and schema are aligned
            entity_labels=['Software', 'Format', 'Concept'],
            threshold=0.3,
        ),
        LLMTopicExtractionStage(),
        EntityMergeStage(fuzzy_threshold=0.85),  # Fuzzy dedup: 'DataBridge' ≈ 'DataBridge AI'
        SchemaFilterStage(schema),  # Light filter: drops only out-of-schema types
    ],
    schema=schema,
)

graph_index = LexicalGraphIndex(
    os.environ['GRAPH_STORE'],
    os.environ['VECTOR_STORE'],
    indexing_config=extraction_config,
)

print('Pipeline stages:')
for i, comp in enumerate(graph_index.extraction_components):
    print(f'  {i+1}. {type(comp).__name__}')

# Load sample.md
docs = SimpleDirectoryReader(input_files=['artifacts/sample.md']).load_data()
print(f'\nLoaded {len(docs)} document(s) from artifacts/sample.md')

# Run extraction and build
graph_index.extract_and_build(docs, show_progress=True)

# Show graph stats
print('\n--- Graph Stats ---')
stats = graph_index.get_stats()
for key in ['source', 'chunk', 'topic', 'statement', 'fact', 'entity']:
    print(f'  {key:12s}: {stats.get(key, 0)}')

print('\n\u2705 Pipeline completed!')
print('   NER and schema aligned on: Software, Format, Concept')
print('   Schema lightly filters out any entity types the LLM invented outside those three')

## Available Stages Reference

| Stage | Purpose | Requires |
|-------|---------|----------|
| `LLMPropositionStage` | Break text into atomic propositions via LLM | — |
| `LocalPropositionStage` | Break text into propositions locally (no LLM cost) | — |
| `LLMTopicExtractionStage` | Extract topics, entities, relationships via LLM | propositions (configurable) |
| `NERExtractionStage` | Find entities using local GLiNER model | `pip install gliner` |
| `EntityMergeStage` | Merge NER entities into LLM-extracted topics (optional fuzzy dedup) | NER output + topics |
| `SchemaFilterStage` | Remove entities/relationships not in schema | topics |
| `BatchLLMPropositionStage` | Batch proposition extraction via Bedrock | `BatchConfig` |
| `BatchTopicExtractionStage` | Batch topic extraction via Bedrock | `BatchConfig` |